### Creaing model for Traditional Breast Cancer dataset using Neural Network with mini Batches using Dataset and DataLoader Classes of PyTorch

In [55]:
# importing requiured modules
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [56]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [57]:
df.shape

(569, 33)

In [58]:
try:
    df.drop(columns=['id', 'Unnamed: 32'], inplace=True)
    print('Sucessfully droped the columns')
except Exception as e:
    print(e)

Sucessfully droped the columns


In [59]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


#### Train/Test split

In [60]:
X_train, X_test, Y_train, Y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

#### Scaling

In [61]:
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

In [62]:
X_train.shape, X_test.shape

((455, 30), (114, 30))

#### Encoding Labels

In [63]:
encoder = LabelEncoder().fit(Y_train)
Y_train = encoder.transform(Y_train)
Y_test = encoder.transform(Y_test)

In [64]:
Y_train[:20]

array([0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1])

#### NumPy Array to PyTorch Tensors

In [79]:
X_train_tensor = torch.from_numpy(X_train).to(torch.float32)
Y_train_tensor = torch.from_numpy(Y_train).to(torch.float32)

X_test_tensor = torch.from_numpy(X_test).to(torch.float32)
Y_test_tensor = torch.from_numpy(Y_test).to(torch.float32)

In [80]:
print(X_train_tensor.shape, Y_train_tensor.shape)
print(X_test_tensor.shape, Y_test_tensor.shape)

torch.Size([455, 30]) torch.Size([455])
torch.Size([114, 30]) torch.Size([114])


In [81]:
# importing Dataset and DataLoader Classes
from torch.utils.data import Dataset, DataLoader

#### Creating custom class to get the data in requied manner

In [82]:
class CustomDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels
        
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [83]:
train_dataset = CustomDataset(X_train_tensor, Y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, Y_test_tensor)

In [84]:
train_dataset[0]

(tensor([-1.2498, -0.9433, -1.1628, -1.0053,  0.7589,  1.1261,  4.1503,  0.7816,
          2.7855,  4.2875,  1.5619,  2.6320,  0.6433,  0.2473,  1.3811,  4.0006,
         11.9713,  6.8082,  1.7811,  9.6968, -1.0944, -1.0292, -1.0880, -0.8867,
         -0.1243,  0.1731,  2.6998,  0.6680,  0.3277,  2.3424]),
 tensor(0.))

#### using DataLoader class to create batches and suffle the data

In [85]:
train_loader = DataLoader(train_dataset, batch_size=10, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=10, shuffle=True)

In [86]:
Y_train.shape, Y_test.shape, 

((455,), (114,))

## Defining Model

In [ ]:
class MySimpleNN(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.linear = nn.Linear(num_features, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, features):
        out = self.linear(features)
        out = self.sigmoid(out)

        return out

In [88]:
## important parameters
epochs = 30
learning_rate = 0.01

In [89]:
# create the model
model = MySimpleNN(X_train_tensor.shape[1])

# define optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# define loss function
loss_function = nn.BCELoss()

### Training Pipeline

In [90]:
# define loop
for epoch in range(epochs):
    for batch_features, batch_labels in train_loader:
        
        # forward pass
        y_pred = model(batch_features)
        
        # loss calculation
        loss = loss_function(y_pred, batch_labels.view(-1, 1))

        # clear gradients
        optimizer.zero_grad()
        
        # backward pass
        loss.backward()
        
        # parameters update
        optimizer.step()

    print(f'Epoch: {epoch+1} | Loss: {loss.item()}')

Epoch: 1 | Loss: 0.533346951007843
Epoch: 2 | Loss: 0.2110992670059204
Epoch: 3 | Loss: 0.24785831570625305
Epoch: 4 | Loss: 0.2355760633945465
Epoch: 5 | Loss: 0.0637686550617218
Epoch: 6 | Loss: 0.2590257525444031
Epoch: 7 | Loss: 0.0738525390625
Epoch: 8 | Loss: 0.03970498591661453
Epoch: 9 | Loss: 0.10566605627536774
Epoch: 10 | Loss: 0.11281641572713852
Epoch: 11 | Loss: 0.176875501871109
Epoch: 12 | Loss: 0.3944384455680847
Epoch: 13 | Loss: 0.043183453381061554
Epoch: 14 | Loss: 0.03500763326883316
Epoch: 15 | Loss: 0.2406013011932373
Epoch: 16 | Loss: 0.05929647013545036
Epoch: 17 | Loss: 0.14696958661079407
Epoch: 18 | Loss: 0.008688656613230705
Epoch: 19 | Loss: 0.041820742189884186
Epoch: 20 | Loss: 0.09391497075557709
Epoch: 21 | Loss: 0.09955578297376633
Epoch: 22 | Loss: 0.07672391831874847
Epoch: 23 | Loss: 0.07267378270626068
Epoch: 24 | Loss: 0.07808578014373779
Epoch: 25 | Loss: 0.40013185143470764
Epoch: 26 | Loss: 0.13145340979099274
Epoch: 27 | Loss: 0.120627805590

### Evaluation

In [91]:
print(batch_labels.shape)

torch.Size([5])


In [95]:
# Model evaluation using the test_loader
model.eval() # Set the model to evaluation mode
accuracy_list = []

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        
        # forward pass
        y_pred = model(batch_features)
        y_pred = (y_pred > 0.8).float()  # Convert probabilities to binary prediction 
        
        # calcuate the accuracy fo the current batch
        batch_accuracy = (y_pred.view(-1) == batch_labels).float().mean().item()
        accuracy_list.append(batch_accuracy)
        
# calcuate overall accuracy
overall_accuracy = sum(accuracy_list)/len(accuracy_list)
print(f'Accuracy: {overall_accuracy:.4f}')

Accuracy: 0.9500
